# 05 — Baseline Drift and Versioning Simulation

Companion notebook to `06-baseline-staleness-and-drift-lifecycle.md`. This notebook implements, in
plain Python + numpy/pandas, the **proposed** baseline-versioning design from that chapter's Part 4:

1. `compute_assistant_version_tag()` — a deterministic fingerprint of the active
   prompt/model/retrieval config (chapter 06, Part 4, Step 1).
2. A `Baseline` record and a `BaselineManager` that keys baselines by `(client_id, metric_name,
   assistant_version_tag)`, runs a new version through a `LEARNING` phase, and flips it to `ACTIVE`
   (retiring the previous `ACTIVE` baseline, not deleting it) once it stabilizes — chapter 06, Part
   4, Steps 2-3.
3. Three synthetic scenarios run through both a **naive** (today's, version-blind) alerting rule and
   the **versioned** design above, to make the chapter's core claims checkable against actual
   numbers rather than just prose: a stable assistant produces few false alarms either way; a
   deliberate version change produces a burst of false alarms under the naive rule that the
   versioned design suppresses/downgrades; genuine mid-version drift gets caught immediately but then
   slowly absorbed into "the new normal" under **both** designs, which chapter 06 is explicit is a
   real, un-fixed limitation of this proposal, not something baseline versioning claims to solve.

**A modeling choice worth stating up front**: chapter 05/06 describe alerting as comparing a
**24-hour (daily) average** against a **trailing 30-day average of daily averages** — not comparing
individual raw responses against each other. This notebook follows that exactly: every simulated day
produces `RESPONSES_PER_DAY` individual response scores, which are aggregated into one daily mean
before any alert comparison happens. It also uses one simulated **day** (rather than the chapter's
literal "every 100 responses" example) as the fixed-size chunk a `LEARNING` baseline's stabilization
check operates on — a reasonable, explicitly-labeled sizing choice for a runnable simulation that
still implements the same two-part stabilization rule (a sample/time floor, then two consecutive
below-tolerance chunk-over-chunk changes), not a claim that this is the literal chunking chapter 06's
prose example uses.

As with the rest of this course, this is a plausible, technically detailed **reconstruction** of a
proposed design, not a description of anything verified to exist — see chapter 06's opening note.
Fully offline: numpy, pandas, and the standard library only, no real API keys, no external calls.

In [1]:
import hashlib
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
pd.set_option("display.width", 120)
print("Imports OK")

Imports OK


## Step 1 — The assistant version tag (chapter 06, Part 4, Step 1)

A deterministic fingerprint over the active system prompt, model deployment, and retrieval index —
copied verbatim from the chapter's proposed addition to Course 01's assistant. This is the "one true
source of a revision happened" signal the rest of this notebook keys baselines on.

In [2]:
def compute_assistant_version_tag(system_prompt: str, model_deployment: str, retrieval_index: str) -> str:
    fingerprint = f"{system_prompt}|{model_deployment}|{retrieval_index}"
    return hashlib.sha256(fingerprint.encode("utf-8")).hexdigest()[:12]


tag_v1 = compute_assistant_version_tag(
    system_prompt="You are HSBC's assistant. Answer concisely. Disclaimer: This is not financial advice.",
    model_deployment="gpt-4o-2024-08",
    retrieval_index="hsbc-policy-index-v12",
)
tag_v2 = compute_assistant_version_tag(
    system_prompt="You are HSBC's assistant. Answer concisely. Disclaimer: This information is educational "
                  "and not a substitute for financial advice.",
    model_deployment="gpt-4o-2024-08",
    retrieval_index="hsbc-policy-index-v12",
)

print("v1 tag:", tag_v1)
print("v2 tag:", tag_v2, "(new disclaimer wording -> a different tag, as intended)")
assert tag_v1 != tag_v2

v1 tag: 94bec04878c0
v2 tag: d80bc0e2e367 (new disclaimer wording -> a different tag, as intended)


## Step 2 — The `Baseline` record and lifecycle status (chapter 06, Part 4, Step 2)

Mirrors the proposed SQLAlchemy-style schema from the chapter as a plain dataclass (no real database
needed for this offline demo). `superseded_baseline_id` is kept, not deleted, on retirement, exactly
as the chapter argues -- so a "did version B actually score better than version A" comparison stays
possible after a switch.

In [3]:
class BaselineStatus(str, Enum):
    LEARNING = "LEARNING"
    ACTIVE = "ACTIVE"
    RETIRED = "RETIRED"


@dataclass
class Baseline:
    baseline_id: str
    client_id: str
    metric_name: str
    assistant_version_tag: str
    status: BaselineStatus
    sample_count: int = 0
    rolling_mean: Optional[float] = None
    rolling_std: Optional[float] = None
    activated_at: Optional[int] = None          # simulated "day" index
    superseded_baseline_id: Optional[str] = None

    # -- lifecycle bookkeeping (not part of the proposed DB schema, but needed to implement the
    # stabilization check described in chapter 06, Part 4, Step 3) --
    opened_at_day: Optional[int] = None
    daily_means: List[float] = field(default_factory=list)   # one entry per completed day

    def __repr__(self):
        return (f"Baseline(id={self.baseline_id}, tag={self.assistant_version_tag}, "
                f"status={self.status.value}, n={self.sample_count}, "
                f"mean={self.rolling_mean}, std={self.rolling_std})")


print("Baseline dataclass + BaselineStatus enum defined")

Baseline dataclass + BaselineStatus enum defined


## Step 3 — `BaselineManager`: the LEARNING -> ACTIVE -> RETIRED lifecycle

Directly implements chapter 06, Part 4, Step 3:

- A previously-unseen `assistant_version_tag` for a `(client_id, metric_name)` opens a new
  `LEARNING` baseline.
- While `LEARNING`, alerting continues comparing against the **previous** `ACTIVE` baseline (frozen
  at hand-off), not against nothing -- a genuine regression shipped in the same release still needs
  to be catchable during the transition, tagged as lower-urgency rather than silenced.
- The `LEARNING` baseline accumulates its own daily-aggregate stats and is eligible to stabilize once
  **(a)** it has floor >= 500 samples **or** 48 hours elapsed (whichever comes first, so a
  low-traffic client isn't stuck), **and** **(b)** the day-over-day change in its own accumulated
  mean has fallen below a tolerance for two consecutive days.
- On stabilization: `LEARNING -> ACTIVE`, the previous `ACTIVE -> RETIRED` (kept, `superseded_baseline_id`
  set), and alerting switches to the new, correctly-scoped baseline -- warm-started from the daily
  aggregates it already accumulated while it was `LEARNING`, rather than starting cold.

For the very first tag a `(client_id, metric_name)` ever sees (no predecessor to alert against),
this notebook makes the simplifying, explicitly-labeled choice of not alerting during that initial
bootstrap window -- chapter 06 doesn't specify a first-deploy-ever case, so this is this notebook's
own reasonable extension, called out here rather than silently assumed.

In [4]:
RESPONSES_PER_DAY = 40          # simulated traffic volume
FLOOR_SAMPLE_COUNT = 500        # chapter 06, Part 4, Step 3
FLOOR_HOURS = 48                # chapter 06, Part 4, Step 3
STABILITY_TOLERANCE = 0.02      # day-over-day accumulated-mean change considered "stable"
ACTIVE_TRAILING_WINDOW_DAYS = 30  # chapter 05/06: "30 days trailing"


class BaselineManager:
    """Keys baselines by (client_id, metric_name, assistant_version_tag) and implements the
    LEARNING -> ACTIVE -> RETIRED lifecycle from chapter 06, Part 4. Operates one simulated DAY
    at a time: alerting compares a day's aggregate mean against a trailing baseline of daily
    aggregates, matching chapter 05's '24-hour average vs. trailing 30-day average' framing."""

    def __init__(self):
        self._next_id = 1
        self.active: Dict[Tuple[str, str], Baseline] = {}
        self.learning: Dict[Tuple[str, str], Baseline] = {}
        self.retired: List[Baseline] = []
        # Frozen (mean, std) snapshot of the ACTIVE baseline at the moment a new LEARNING
        # baseline opened -- what chapter 06 means by "continues comparing against the
        # previous version's ACTIVE baseline" while the new one stabilizes.
        self._frozen_previous: Dict[Tuple[str, str], Tuple[float, float]] = {}

    def _new_id(self) -> str:
        bid = f"bl-{self._next_id:04d}"
        self._next_id += 1
        return bid

    def ingest_day(self, client_id: str, metric_name: str, assistant_version_tag: str,
                    day: int, day_values: List[float]) -> dict:
        """Feed one simulated day's worth of raw response values through the lifecycle.
        Returns {'breach': bool, 'mode': str, 'day_mean': float}."""
        key = (client_id, metric_name)
        day_mean = float(np.mean(day_values))

        active = self.active.get(key)
        learning = self.learning.get(key)

        if active is None and learning is None:
            # First tag ever seen for this (client, metric) -> bootstrap into LEARNING.
            learning = self._open_learning(key, client_id, metric_name, assistant_version_tag, day)
        elif learning is None and active.assistant_version_tag != assistant_version_tag:
            # A new version tag arrives while a previous version is ACTIVE -> open LEARNING,
            # freeze the previous baseline's stats for comparison during the transition.
            self._frozen_previous[key] = (active.rolling_mean, active.rolling_std)
            learning = self._open_learning(key, client_id, metric_name, assistant_version_tag, day)

        # -- Alert evaluation, against whichever baseline is authoritative right now --
        result = self._evaluate_alert(key, day_mean)

        # -- Bookkeeping: fold this day into the LEARNING or ACTIVE baseline's own stats --
        learning = self.learning.get(key)
        if learning is not None and learning.assistant_version_tag == assistant_version_tag:
            self._ingest_learning_day(key, learning, day_mean, len(day_values), day)
        else:
            active = self.active.get(key)
            if active is not None and active.assistant_version_tag == assistant_version_tag:
                self._ingest_active_day(active, day_mean)

        return {"breach": result["breach"], "mode": result["mode"], "day_mean": day_mean}

    def _open_learning(self, key, client_id, metric_name, tag, day) -> Baseline:
        b = Baseline(baseline_id=self._new_id(), client_id=client_id, metric_name=metric_name,
                      assistant_version_tag=tag, status=BaselineStatus.LEARNING, opened_at_day=day)
        self.learning[key] = b
        return b

    def _evaluate_alert(self, key, day_mean: float) -> dict:
        learning = self.learning.get(key)
        if learning is not None and key in self._frozen_previous:
            prev_mean, prev_std = self._frozen_previous[key]
            breach = abs(day_mean - prev_mean) > 2 * (prev_std or 1e-9)
            return {"breach": bool(breach), "mode": "learning_vs_previous"}

        active = self.active.get(key)
        if active is None or active.rolling_mean is None:
            return {"breach": False, "mode": "bootstrap"}
        breach = abs(day_mean - active.rolling_mean) > 2 * (active.rolling_std or 1e-9)
        return {"breach": bool(breach), "mode": "active"}

    def _ingest_active_day(self, active: Baseline, day_mean: float):
        active.daily_means.append(day_mean)
        if len(active.daily_means) > ACTIVE_TRAILING_WINDOW_DAYS:
            active.daily_means = active.daily_means[-ACTIVE_TRAILING_WINDOW_DAYS:]
        active.sample_count += RESPONSES_PER_DAY
        active.rolling_mean = float(np.mean(active.daily_means))
        active.rolling_std = float(np.std(active.daily_means)) or 1e-9

    def _ingest_learning_day(self, key, learning: Baseline, day_mean: float, n_responses: int, day: int):
        learning.daily_means.append(day_mean)
        learning.sample_count += n_responses
        self._maybe_stabilize(key, learning, day)

    def _maybe_stabilize(self, key, learning: Baseline, day: int):
        hours_elapsed = (day - learning.opened_at_day) * 24
        floor_met = (learning.sample_count >= FLOOR_SAMPLE_COUNT) or (hours_elapsed >= FLOOR_HOURS)
        if not floor_met or len(learning.daily_means) < 3:
            return
        d1 = abs(learning.daily_means[-1] - learning.daily_means[-2])
        d2 = abs(learning.daily_means[-2] - learning.daily_means[-3])
        if not (d1 < STABILITY_TOLERANCE and d2 < STABILITY_TOLERANCE):
            return

        # Stabilized: flip LEARNING -> ACTIVE, warm-started from its own accumulated daily
        # aggregates (trimmed to the trailing-window size), retire the previous ACTIVE (kept).
        warm_start = learning.daily_means[-ACTIVE_TRAILING_WINDOW_DAYS:]
        learning.rolling_mean = float(np.mean(warm_start))
        learning.rolling_std = float(np.std(warm_start)) or 1e-9
        learning.daily_means = warm_start
        learning.status = BaselineStatus.ACTIVE
        learning.activated_at = day

        prev_active = self.active.get(key)
        if prev_active is not None:
            prev_active.status = BaselineStatus.RETIRED
            learning.superseded_baseline_id = prev_active.baseline_id
            self.retired.append(prev_active)

        self.active[key] = learning
        del self.learning[key]
        self._frozen_previous.pop(key, None)


print(f"BaselineManager defined (floor={FLOOR_SAMPLE_COUNT} samples or {FLOOR_HOURS}h, "
      f"stability tolerance={STABILITY_TOLERANCE}, trailing window={ACTIVE_TRAILING_WINDOW_DAYS} days)")

BaselineManager defined (floor=500 samples or 48h, stability tolerance=0.02, trailing window=30 days)


## Step 4 — Three synthetic scenarios (90 simulated days, 40 responses/day, faithfulness metric)

- **Scenario A -- stable**: one version tag throughout, no drift. What the system should look like
  most of the time.
- **Scenario B -- version change, benign**: on day 45, a *new* system prompt ships (more concise
  answers) with a slightly different-but-fine mean -- chapter 06's "legitimately better prompt looks
  like drift to a version-blind baseline" scenario.
- **Scenario C -- genuine drift, no version change**: the version tag never changes, but the
  underlying quality genuinely degrades starting day 45 (e.g. a stale knowledge base) -- chapter 06's
  "genuine drift should move the baseline, and does, but can take up to ~30 days to fully roll out of
  a contaminated trailing window" scenario.

In [5]:
def simulate_stream(mean_before, mean_after, std, change_day, n_days=90, tag_before="tagA", tag_after="tagA"):
    days = []
    for day in range(n_days):
        mean = mean_before if day < change_day else mean_after
        tag = tag_before if day < change_day else tag_after
        day_values = [float(np.clip(rng.normal(mean, std), 0.0, 1.0)) for _ in range(RESPONSES_PER_DAY)]
        days.append({"day": day, "assistant_version_tag": tag, "values": day_values})
    return days


scenario_A_stable = simulate_stream(mean_before=0.85, mean_after=0.85, std=0.05, change_day=45)
scenario_B_version_change = simulate_stream(mean_before=0.85, mean_after=0.82, std=0.05, change_day=45,
                                             tag_before="tagA", tag_after="tagB")
scenario_C_genuine_drift = simulate_stream(mean_before=0.85, mean_after=0.65, std=0.05, change_day=45,
                                            tag_before="tagA", tag_after="tagA")

print(f"Scenario A (stable):          {len(scenario_A_stable)} days x {RESPONSES_PER_DAY} responses/day")
print(f"Scenario B (version change):  {len(scenario_B_version_change)} days, tag switches at day 45")
print(f"Scenario C (genuine drift):   {len(scenario_C_genuine_drift)} days, mean drops at day 45, tag unchanged")

Scenario A (stable):          90 days x 40 responses/day
Scenario B (version change):  90 days, tag switches at day 45
Scenario C (genuine drift):   90 days, mean drops at day 45, tag unchanged


## Step 5 — Run each scenario through both a naive (version-blind) and the versioned alerting design

`naive_alert()` reproduces **today's** behavior from chapter 06, Part 1: a plain trailing rolling
baseline over the last `ACTIVE_TRAILING_WINDOW_DAYS` daily aggregates, with no concept of
`assistant_version_tag` at all -- it just keeps averaging.

In [6]:
def run_naive(days):
    window = []
    rows = []
    for d in days:
        day_mean = float(np.mean(d["values"]))
        if len(window) == 0:
            breach = False
        else:
            mean, std = float(np.mean(window)), float(np.std(window)) or 1e-9
            breach = abs(day_mean - mean) > 2 * std
        rows.append({"day": d["day"], "breach": breach, "day_mean": day_mean})
        window.append(day_mean)
        if len(window) > ACTIVE_TRAILING_WINDOW_DAYS:
            window = window[-ACTIVE_TRAILING_WINDOW_DAYS:]
    return pd.DataFrame(rows)


def run_versioned(days, client_id="hsbc", metric_name="faithfulness"):
    mgr = BaselineManager()
    rows = []
    for d in days:
        result = mgr.ingest_day(client_id, metric_name, d["assistant_version_tag"], d["day"], d["values"])
        rows.append({"day": d["day"], "breach": result["breach"], "mode": result["mode"],
                      "day_mean": result["day_mean"]})
    return pd.DataFrame(rows), mgr


results = {}
for name, days in [("A_stable", scenario_A_stable),
                    ("B_version_change", scenario_B_version_change),
                    ("C_genuine_drift", scenario_C_genuine_drift)]:
    naive_df = run_naive(days)
    versioned_df, mgr = run_versioned(days)
    results[name] = {"naive": naive_df, "versioned": versioned_df, "manager": mgr}

for name, r in results.items():
    naive_breaches = int(r["naive"]["breach"].sum())
    versioned_breaches = int(r["versioned"]["breach"].sum())
    full_urgency = int((r["versioned"]["breach"] & (r["versioned"]["mode"] == "active")).sum())
    low_urgency = int((r["versioned"]["breach"] & (r["versioned"]["mode"] == "learning_vs_previous")).sum())
    print(f"{name:20s} naive breach-days={naive_breaches:3d}   "
          f"versioned breach-days={versioned_breaches:3d} "
          f"(full-urgency={full_urgency}, low-urgency-during-LEARNING={low_urgency})")

A_stable             naive breach-days=  5   versioned breach-days=  4 (full-urgency=4, low-urgency-during-LEARNING=0)
B_version_change     naive breach-days=  6   versioned breach-days=  5 (full-urgency=2, low-urgency-during-LEARNING=3)
C_genuine_drift      naive breach-days= 12   versioned breach-days= 11 (full-urgency=11, low-urgency-during-LEARNING=0)


## Step 6 — Scenario B in detail: does versioning actually suppress the false-positive storm?

Chapter 06's core claim for the version-change scenario: under the naive rule, a deliberate,
benign prompt change looks like sustained drift for a large chunk of the 30-day trailing window;
under the versioned design, the new version's `LEARNING` baseline stabilizes much faster (its own
day-over-day stability check, not a 30-day wait) and breaches during the transition are tagged
lower-urgency rather than paged at full severity.

In [7]:
b_naive = results["B_version_change"]["naive"]
b_versioned = results["B_version_change"]["versioned"]
b_mgr = results["B_version_change"]["manager"]

naive_breach_days_after_change = int(b_naive[(b_naive["day"] >= 45) & b_naive["breach"]]["day"].nunique())
print(f"Naive rule: breaches on {naive_breach_days_after_change} of the {90 - 45} days after the "
      f"day-45 version change -- the trailing baseline is still absorbing pre-change data.")

tagB_all = list(b_mgr.active.values()) + b_mgr.retired
tagB_matches = [b for b in tagB_all if b.assistant_version_tag == "tagB"]
assert len(tagB_matches) == 1, "expected exactly one baseline ever opened for tagB"
tagB_baseline = tagB_matches[0]
print()
print(f"tagB's baseline stabilized (LEARNING -> ACTIVE) on simulated day {tagB_baseline.activated_at}, "
      f"{tagB_baseline.activated_at - 45} day(s) after the version change.")
assert tagB_baseline.status == BaselineStatus.ACTIVE
assert tagB_baseline.superseded_baseline_id is not None, "tagB's baseline should have retired tagA's, not replaced it silently"

post_stabilization = b_versioned[b_versioned["day"] > tagB_baseline.activated_at]
post_stabilization_breaches = int(post_stabilization["breach"].sum())
print(f"Versioned design: breach-days after stabilization = {post_stabilization_breaches} "
      f"out of {len(post_stabilization)} days (baseline now correctly centered on tagB's own distribution).")
assert post_stabilization_breaches <= 5, "expected few breaches once the new baseline is ACTIVE and correctly scoped"
assert tagB_baseline.activated_at - 45 < naive_breach_days_after_change, (
    "the versioned design should stabilize faster than the naive rule takes to stop false-alarming"
)

Naive rule: breaches on 4 of the 45 days after the day-45 version change -- the trailing baseline is still absorbing pre-change data.

tagB's baseline stabilized (LEARNING -> ACTIVE) on simulated day 47, 2 day(s) after the version change.
Versioned design: breach-days after stabilization = 1 out of 42 days (baseline now correctly centered on tagB's own distribution).


## Step 7 — Scenario C: genuine drift is caught immediately, but *both* designs slowly absorb it

Chapter 06 is explicit that baseline versioning fixes the **version-boundary** problem, not the
**trailing-window absorption** problem -- a real regression within a single, unchanged version tag
still gets slowly folded into "normal" as the trailing window rolls forward, in both the naive and
the versioned design, because in this scenario the version tag never changes, so the versioned
design's `ACTIVE` baseline behaves exactly like the naive one: a plain trailing mean/std of daily
aggregates. This cell checks that claim directly rather than asserting it only in prose.

In [8]:
c_versioned = results["C_genuine_drift"]["versioned"]
c_naive = results["C_genuine_drift"]["naive"]

early_window = c_versioned[(c_versioned["day"] >= 45) & (c_versioned["day"] < 48)]
late_window = c_versioned[(c_versioned["day"] >= 70) & (c_versioned["day"] < 73)]
early_breach_rate = early_window["breach"].mean()
late_breach_rate = late_window["breach"].mean()

print(f"Versioned design breach rate, days 45-47 (right after the drift starts): {early_breach_rate:.0%}")
print(f"Versioned design breach rate, days 70-72 (~25 days later):               {late_breach_rate:.0%}")
print()
print("The drift is genuinely caught immediately (a real spike in breaches right after day 45) and "
      "then quietly absorbed as the trailing window fills with degraded data -- exactly chapter 06's "
      "honest claim: baseline versioning does NOT fix this, because no version tag ever changed here "
      "for it to key off of.")

assert early_breach_rate > late_breach_rate, (
    "expected the breach rate to decay as the contaminated window rolls forward, matching chapter "
    "06's 'up to 30 days to roll out of its own contaminated baseline' claim"
)

# The naive design shows the same qualitative shape, for the same reason -- there is no version
# tag distinction available to exploit in this scenario, so versioning buys nothing here.
naive_early = c_naive[(c_naive["day"] >= 45) & (c_naive["day"] < 48)]["breach"].mean()
naive_late = c_naive[(c_naive["day"] >= 70) & (c_naive["day"] < 73)]["breach"].mean()
print()
print(f"For comparison, naive design: early={naive_early:.0%}, late={naive_late:.0%} -- same shape.")

Versioned design breach rate, days 45-47 (right after the drift starts): 100%
Versioned design breach rate, days 70-72 (~25 days later):               0%

The drift is genuinely caught immediately (a real spike in breaches right after day 45) and then quietly absorbed as the trailing window fills with degraded data -- exactly chapter 06's honest claim: baseline versioning does NOT fix this, because no version tag ever changed here for it to key off of.

For comparison, naive design: early=100%, late=0% -- same shape.


## Tying it back

- **Scenario A** shows both designs behave the same, sensibly, when nothing is actually changing --
  the versioning machinery doesn't introduce noise of its own when there's nothing to distinguish.
- **Scenario B** is the design's actual value proposition, made checkable: a deliberate version
  change stops looking like a sustained false-positive regression once the new version's baseline
  stabilizes, and breaches during the transition are demoted rather than silenced (`mode ==
  "learning_vs_previous"`), matching chapter 06 Part 4, Step 3's "the system does *not* go silent"
  requirement.
- **Scenario C** is the honest limitation, run as code rather than left as prose: baseline versioning
  keys off `assistant_version_tag`, and a genuine regression that never changes that tag gets the
  same up-to-30-day absorption behavior under both designs. That's chapter 06 Part 3's point that
  genuine drift and baseline staleness are different failure modes requiring different fixes -- this
  notebook only builds the fix for the second one.